# Groups

In [8]:
import json
import pandas as pd

path = "/Users/dmk6603/Documents/ransom_victims/1-ransomware.live_data/data/groups.json"

# carica il JSON come dict
with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

# prendi la lista di gruppi e trasformala in dataframe
df = pd.json_normalize(data["groups"])

# (opzionale) aggiungi i metadata in colonne, se ti servono
df["client"] = data.get("client")

# salva in CSV
out = "/Users/dmk6603/Documents/ransom_victims/1-ransomware.live_data/data/groups.csv"
df.to_csv(out, index=False)

print(df.head())
print("Saved:", out)

         group altname  victims              client
0         0apt    None        0  dmk6603@utulsa.edu
1        0mega    None        7  dmk6603@utulsa.edu
2        8base    None      455  dmk6603@utulsa.edu
3  abrahams_ax    None        0  dmk6603@utulsa.edu
4        abyss    None       86  dmk6603@utulsa.edu
Saved: /Users/dmk6603/Documents/ransom_victims/1-ransomware.live_data/data/groups.csv


# get info for all groups

In [9]:
import os
import json
import time
import re
from pathlib import Path

import requests


# ====== CONFIG ======
API_KEY = "8f316c1d-730e-426b-b72e-006066919222"  # esportala in env: export RWL_API_KEY="..." X-API-KEY: 8f316c1d-730e-426b-b72e-006066919222
BASE_URL = "https://api-pro.ransomware.live/groups"

# input: file creato prima
GROUPS_JSON_PATH = Path("/Users/dmk6603/Documents/ransom_victims/1-ransomware.live_data/data/groups.json")

# output folder
OUT_DIR = Path("/Users/dmk6603/Documents/ransom_victims/1-ransomware.live_data/data/groups_detailed")

# politiche di retry
TIMEOUT = 30
MAX_RETRIES = 5
SLEEP_BETWEEN = 0.25  # per non martellare l’API
# ====================


def sanitize_filename(name: str) -> str:
    """
    Permette solo caratteri safe per filename.
    """
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9._-]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name or "unknown"


def load_group_names_from_groups_json(path: Path) -> list[str]:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    groups = data.get("groups", [])
    names = []
    for g in groups:
        n = g.get("group")
        if n:
            names.append(str(n).strip())
    # dedup preservando ordine
    seen = set()
    out = []
    for n in names:
        if n not in seen:
            out.append(n)
            seen.add(n)
    return out


def fetch_group_detail(session: requests.Session, group: str) -> dict:
    url = f"{BASE_URL}/{group}"
    headers = {
        "accept": "application/json",
        "X-API-KEY": API_KEY,
    }

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, headers=headers, timeout=TIMEOUT)

            # Rate limit
            if r.status_code == 429:
                retry_after = r.headers.get("Retry-After")
                wait = int(retry_after) if (retry_after and retry_after.isdigit()) else (2 * attempt)
                time.sleep(wait)
                continue

            # Auth / permessi
            if r.status_code in (401, 403):
                raise RuntimeError(f"Auth error {r.status_code}: controlla API key/permessi")

            # Not found: gruppo non esiste più o naming mismatch
            if r.status_code == 404:
                return {"_error": "not_found", "_group": group}

            r.raise_for_status()
            return r.json()

        except Exception as e:
            last_err = e
            # backoff semplice
            time.sleep(0.75 * attempt)

    raise RuntimeError(f"Failed after {MAX_RETRIES} retries for group '{group}': {last_err}")


def main():
    if not API_KEY:
        raise SystemExit(
            "Manca RWL_API_KEY. Esempio: export RWL_API_KEY='...'\n"
            "Poi rilancia lo script."
        )

    if not GROUPS_JSON_PATH.exists():
        raise SystemExit(f"File non trovato: {GROUPS_JSON_PATH}")

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    group_names = load_group_names_from_groups_json(GROUPS_JSON_PATH)
    print(f"Found {len(group_names)} groups")

    ok, skipped, failed = 0, 0, 0

    with requests.Session() as session:
        for i, group in enumerate(group_names, 1):
            safe = sanitize_filename(group)
            out_path = OUT_DIR / f"{safe}.json"

            # se già esiste, salta (utile per riprendere)
            if out_path.exists() and out_path.stat().st_size > 0:
                skipped += 1
                continue

            print(f"[{i}/{len(group_names)}] Downloading {group} -> {out_path.name}")

            try:
                data = fetch_group_detail(session, group)

                # salva JSON pretty
                with out_path.open("w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)

                ok += 1
                time.sleep(SLEEP_BETWEEN)

            except Exception as e:
                failed += 1
                # salva errore per debug
                err_path = OUT_DIR / f"{safe}.__error__.txt"
                err_path.write_text(str(e), encoding="utf-8")
                print(f"  ERROR: {e}")

    print(f"Done. ok={ok}, skipped={skipped}, failed={failed}")
    print(f"Output folder: {OUT_DIR}")


if __name__ == "__main__":
    main()

Found 319 groups
[1/319] Downloading 0apt -> 0apt.json
[2/319] Downloading 0mega -> 0mega.json
[3/319] Downloading 8base -> 8base.json
[4/319] Downloading abrahams_ax -> abrahams_ax.json
[5/319] Downloading abyss -> abyss.json
[6/319] Downloading adminlocker -> adminlocker.json
[7/319] Downloading againstthewest -> againstthewest.json
[8/319] Downloading agl0bgvycg -> agl0bgvycg.json
[9/319] Downloading akira -> akira.json
[10/319] Downloading ako -> ako.json
[11/319] Downloading alphalocker -> alphalocker.json
[12/319] Downloading alphv -> alphv.json
[13/319] Downloading anubis -> anubis.json
[14/319] Downloading apos -> apos.json
[15/319] Downloading apt73 -> apt73.json
[16/319] Downloading arcusmedia -> arcusmedia.json
[17/319] Downloading argonauts -> argonauts.json
[18/319] Downloading arkana -> arkana.json
[19/319] Downloading arvinclub -> arvinclub.json
[20/319] Downloading atomsilo -> atomsilo.json
[21/319] Downloading avaddon -> avaddon.json
[22/319] Downloading avos -> avos.j